## Executive Summary

**Purpose:** Turn a ratio into a meshing helical pair: tooth counts, helix angle, centre distance, and the split of the profile shift between pinion and gear.

**What it does:** `pair_tooth_counts` lists pinion and gear tooth counts near a target ratio, hunting-tooth pairs first. `HELIX_PRIORITIES` and `helix_angles` turn a design priority (quiet running, balanced, or low axial thrust) into the helix angles to try. `split_shift_sum` shares a shift sum between the two gears for balanced sliding and strength. `HelicalPair.build` makes the pair, from a centre distance if one is fixed, and `HelicalPair` gives the speeds, the mesh frequency, the tooth forces and readable checks.

**Units:** mm, N, N·m, rpm.

## How It Fits Into the Helical Gear Workflow

**Upstream:** `gears.helical.geometry` (one gear and the mesh checks).

**Downstream:** `gears.helical.rating` (rates a `HelicalPair`), `gears.helical.components` (bearing loads from `HelicalPair.forces`), `gears.helical.design` (searches over the choices made here).

In [ ]:
#| default_exp gears.helical.layout

In [ ]:
#| export
from __future__ import annotations

import math
from dataclasses import dataclass

from emachines.gears.helical.geometry import (HelicalGear, HelicalMesh, helical_mesh_check, helical_centre_distance,
                                              helical_shift_sum)

## Tooth Counts

A single external stage has ratio $u = z_2/z_1$ and the gear turns the other way. Ratios up to about 5 to 6 are practical in one stage. Above that the gear gets large next to the pinion, and two stages are smaller.

Two rules choose among the counts that meet the ratio:

- **Hunting teeth:** when $z_1$ and $z_2$ have no common factor, every pinion tooth meets every gear tooth in turn. Wear spreads evenly, and a local defect does not keep hitting the same mate tooth.
- **Pinion size:** fewer pinion teeth give a smaller pair for the same module. A helical pinion can go lower than a spur one before it undercuts (about 14 teeth at 15° without shift, and fewer with positive shift). Down to 12 is used for compact drives.

`pair_tooth_counts` returns every $(z_1, z_2)$ within the tolerance. They are sorted by ratio error in 0.5 % steps, then hunting pairs first, then by pinion teeth.

In [ ]:
#| export
@dataclass(frozen=True)
class PairCounts:
    "One pinion/gear tooth count combination."
    z1: int
    z2: int
    ratio: float
    error: float
    hunting: bool


def pair_tooth_counts(ratio: float, tol: float = 0.02, z1_range: tuple[int, int] = (12, 40),
                      z2_max: int = 250) -> list[PairCounts]:
    "Pinion and gear tooth counts within `tol` of `ratio` (see above), best first."
    if ratio < 1:
        raise ValueError("give the ratio as z2/z1 ≥ 1 (pinion drives)")
    out = []
    for z1 in range(z1_range[0], z1_range[1] + 1):
        for z2 in range(max(z1, math.floor(ratio * z1 * (1 - tol))), math.ceil(ratio * z1 * (1 + tol)) + 1):
            if z2 > z2_max:
                continue
            u = z2 / z1
            err = (u - ratio) / ratio
            if abs(err) <= tol + 1e-12:
                out.append(PairCounts(z1, z2, u, err, math.gcd(z1, z2) == 1))
    out.sort(key=lambda c: (round(abs(c.error) / 0.005), not c.hunting, c.z1))
    return out

In [ ]:
for c in pair_tooth_counts(3.0, tol=0.02, z1_range=(14, 20)):
    print(f"{c.z1}/{c.z2}: u = {c.ratio:.4f} ({c.error * 100:+.2f} %), hunting {c.hunting}")

In [ ]:
#| hide
_c = pair_tooth_counts(3.0, tol=0.02, z1_range=(14, 20))
assert all(abs(c.error) <= 0.02 + 1e-12 for c in _c)
assert _c[0].error == 0 or abs(_c[0].error) < 0.005
assert all(c.hunting == (math.gcd(c.z1, c.z2) == 1) for c in _c)
assert any(c.z1 == 17 and c.z2 == 52 for c in _c)          # 3.059, hunting
assert not any(c.z1 == 15 and c.z2 == 45 and c.hunting for c in _c)

## Choosing the Helix Angle

The helix angle is a trade-off, so it is chosen from what the drive is for:

| Priority | β tried | Why |
|:--|:--:|:--|
| `quiet` | 20°–30° | High overlap. Pick the face width for a whole-number $\varepsilon_\beta$ so the total contact line length stays constant, which minimises mesh excitation. Costs axial thrust ($F_a = F_t\tan\beta_w$, 36–58 % of $F_t$) and needs bearings that take it. |
| `balanced` | 12°–20° | The usual industrial choice. $\varepsilon_\beta \ge 1$ comes at a modest face width, the axial force is 21–36 % of $F_t$, and deep groove ball bearings take it. |
| `low_thrust` | 8°–12° | Keeps the axial force small (14–21 % of $F_t$) for light bearings, plastic housings, or a motor shaft whose bearings must not see much thrust. Needs a wider face for $\varepsilon_\beta \ge 1$. |

Other considerations:

- The tooth gets stronger in bending with helix angle up to about 30° (factor $Y_\beta$), and the contact stress falls through $Z_\varepsilon$ once $\varepsilon_\beta \ge 1$. Above about 25° the transverse contact ratio falls noticeably and the base pitch error matters more.
- **Hand:** the pinion is usually right-hand. Choose the hand so the axial force of the normal driving direction pushes the pinion toward its located bearing, or toward the motor if the motor bearings are designed for it.
- **Exact angles are free:** a hob cuts any helix angle, so when the centre distance is fixed the angle can be set to hit it with zero shift sum (see below).

In [ ]:
#| export
HELIX_PRIORITIES: dict[str, tuple[float, ...]] = {
    "quiet": (20.0, 22.5, 25.0, 30.0),
    "balanced": (15.0, 12.0, 18.0, 20.0),
    "low_thrust": (10.0, 8.0, 12.0),
}

HELIX_REASONS: dict[str, str] = {
    "quiet": "quiet running: high overlap with a whole-number ε_β, at the cost of more axial thrust",
    "balanced": "balanced: ε_β ≥ 1 at a modest face width, axial thrust that deep groove ball bearings take",
    "low_thrust": "low axial thrust: small helix angle for light bearings or a motor shaft, with a wider face for ε_β ≥ 1",
}


def helix_angles(priority: str) -> tuple[float, ...]:
    "Helix angles [°] to try for a design priority ('quiet', 'balanced' or 'low_thrust'), preferred first."
    if priority not in HELIX_PRIORITIES:
        raise ValueError(f"priority must be one of {list(HELIX_PRIORITIES)}")
    return HELIX_PRIORITIES[priority]


def face_width_for_overlap(mn: float, beta: float, eps_beta: float) -> float:
    "Face width [mm] that gives overlap ratio eps_beta: b = ε_β π m_n / sin β."
    if beta <= 0:
        raise ValueError("a spur gear has no overlap")
    return eps_beta * math.pi * mn / math.sin(beta)

In [ ]:
#| hide
assert helix_angles("balanced")[0] == 15.0
try:
    helix_angles("fast")
    raise AssertionError
except ValueError:
    pass
from emachines.gears.helical.geometry import overlap_ratio
assert math.isclose(overlap_ratio(face_width_for_overlap(1.5, 0.3, 2.0), 1.5, 0.3), 2.0)

## Profile Shift

### Undercut limit

A hobbed helical gear with $z$ teeth undercuts unless (from the limit in `gears.helical.geometry`)

$$x \ge x_{min} = h_{fP}^* - \rho_{fP}^*(1-\sin\alpha_n) - \frac{z\sin^2\alpha_t}{2\cos\beta} .$$

### Splitting the shift sum

The shift sum $x_1 + x_2$ comes from the centre distance. Its split between pinion and gear is a free choice. A positive shift on the pinion thickens its root and moves contact away from its fillet, where the sliding is fastest. A common rule (Niemann, also given in ISO/TR 4467 and DIN 3992) balances the specific sliding and the root strength of the two gears in terms of the virtual tooth counts:

$$x_1 = \frac{x_1 + x_2}{2} + \left(\frac12 - \frac{x_1 + x_2}{2}\right)\frac{\lg(z_{n2}/z_{n1})}{\lg(z_{n1}z_{n2}/100)},\qquad x_2 = (x_1+x_2) - x_1 .$$

Each gear is then raised to its undercut limit plus 0.05 if needed, and the other gear gives the shift back.

In [ ]:
#| export
def undercut_shift_helical(z: int, beta: float, hf: float = 1.25, rho_f: float = 0.38,
                           alpha_n: float = math.radians(20.0)) -> float:
    "Smallest profile shift that avoids undercut of a hobbed helical gear with z teeth."
    at = math.atan(math.tan(alpha_n) / math.cos(beta))
    return hf - rho_f * (1 - math.sin(alpha_n)) - z * math.sin(at) ** 2 / (2 * math.cos(beta))


def split_shift_sum(z1: int, z2: int, x_sum: float, beta: float,
                    alpha_n: float = math.radians(20.0), margin: float = 0.05) -> tuple[float, float]:
    "Share x_sum between pinion and gear (Niemann rule above), keeping both clear of undercut."
    bb = math.asin(math.sin(beta) * math.cos(alpha_n))
    k = 1 / (math.cos(bb) ** 2 * math.cos(beta))
    zn1, zn2 = z1 * k, z2 * k
    den = max(math.log10(zn1 * zn2 / 100), 0.3)
    x1 = x_sum / 2 + (0.5 - x_sum / 2) * math.log10(zn2 / zn1) / den
    lo1 = undercut_shift_helical(z1, beta, alpha_n=alpha_n) + margin
    lo2 = undercut_shift_helical(z2, beta, alpha_n=alpha_n) + margin
    x1 = max(x1, lo1)
    x1 = min(x1, x_sum - lo2) if x_sum - lo2 >= lo1 else x1
    return x1, x_sum - x1

In [ ]:
#| hide
_x1, _x2 = split_shift_sum(17, 52, 0.2, math.radians(15))
assert math.isclose(_x1 + _x2, 0.2) and _x1 > _x2
assert _x1 >= undercut_shift_helical(17, math.radians(15)) + 0.05
_x1, _x2 = split_shift_sum(12, 36, 0.0, math.radians(15))      # small pinion: raised past its undercut limit
assert _x1 >= undercut_shift_helical(12, math.radians(15)) + 0.05 - 1e-12 and math.isclose(_x1 + _x2, 0.0)
_x1, _x2 = split_shift_sum(30, 30, 0.4, math.radians(15))       # equal gears share equally
assert math.isclose(_x1, 0.2) and math.isclose(_x2, 0.2)

## The Pair

`HelicalPair.build` makes the two gears:

1. **Shift sum.** If a centre distance $a_w$ is given, the shift sum that reaches it comes from `helical_shift_sum`. If not, the reference centre distance is rounded **up** to a tidy value (0.1 mm below 20 mm, 0.5 mm below 100 mm, 1 mm above) and the small positive shift sum that reaches it is used. A round number is easier to machine and to inspect in the housing.
2. **Split** with `split_shift_sum`, unless the shifts are given.
3. **Tip shortening.** A positive shift sum moves the gears apart by less than $(x_1+x_2)m_n$, so the tips are shortened by $k^* = (x_1+x_2) - (a_w - a)/m_n$ to keep the standard tip clearance (ISO 21771).
4. **Checks** with `helical_mesh_check`.

The pinion is right-hand by default and the gear left-hand.

**Tooth forces** at the operating pitch circle $d_{w1} = 2a_w/(1+u)$, with pinion torque $T_1$:

$$F_t = \frac{2000\,T_1}{d_{w1}},\qquad F_r = F_t\tan\alpha_{wt},\qquad F_a = F_t\tan\beta_w .$$

The axial force flips with the direction of torque, so a drive that runs both ways or brakes must locate both shafts against axial force in both directions.

In [ ]:
#| export
def tidy_centre_distance(a: float) -> float:
    "Round a centre distance [mm] up: 0.1 mm below 20 mm, 0.5 mm below 100 mm, else 1 mm."
    step = 0.1 if a < 20 else (0.5 if a < 100 else 1.0)
    return math.ceil(a / step - 1e-9) * step


@dataclass(frozen=True)
class HelicalPair:
    """
    An external helical gear pair, pinion driving.

    pinion, gear : HelicalGear
    mesh         : HelicalMesh  Geometry checks of the mesh
    """
    pinion: HelicalGear
    gear: HelicalGear
    mesh: HelicalMesh

    @classmethod
    def build(cls, z1: int, z2: int, mn: float, beta: float, b: float, a_w: float | None = None,
              x1: float | None = None, x2: float | None = None, alpha_n: float = math.radians(20.0),
              hand: str = "right", b2: float | None = None) -> "HelicalPair":
        """
        Build the pair as described above. beta [rad]; b: pinion face width [mm] (gear: b2, default b);
        a_w: operating centre distance [mm] (None: tidy value just above the reference centre distance);
        hand: hand of the pinion.
        """
        a0 = (z1 + z2) * mn / math.cos(beta) / 2
        if x1 is not None and x2 is not None:
            xs = x1 + x2
        else:
            xs = helical_shift_sum(z1, z2, mn, beta, a_w if a_w is not None else tidy_centre_distance(a0), alpha_n)
            if x1 is None:
                x1, x2 = split_shift_sum(z1, z2, xs, beta, alpha_n)
            else:
                x2 = xs - x1
        other = "left" if hand == "right" else "right"
        p = HelicalGear(z=z1, mn=mn, beta=beta, x=x1, b=b, hand=hand, alpha_n=alpha_n, name="pinion")
        g = HelicalGear(z=z2, mn=mn, beta=beta, x=x2, b=b if b2 is None else b2, hand=other if beta > 0 else hand,
                        alpha_n=alpha_n, name="gear")
        aw = helical_centre_distance(p, g)
        k = xs - (aw - a0) / mn
        if k > 1e-9:
            p, g = p.with_(k_tip=k), g.with_(k_tip=k)
        return cls(p, g, helical_mesh_check(p, g))

    # --- kinematics ------------------------------------------------------------
    @property
    def ratio(self) -> float:
        "Gear ratio u = z2 / z1 (= n_pinion / n_gear)."
        return self.gear.z / self.pinion.z

    @property
    def a_w(self) -> float:
        return self.mesh.a_w

    @property
    def d_w1(self) -> float:
        "Operating pitch diameter of the pinion [mm]."
        return 2 * self.a_w / (1 + self.ratio)

    @property
    def d_w2(self) -> float:
        return self.d_w1 * self.ratio

    def speeds(self, n1: float) -> dict[str, float]:
        "Speeds [rpm]; the gear turns the other way (negative)."
        return {"pinion": n1, "gear": -n1 / self.ratio}

    def mesh_frequency(self, n1: float) -> float:
        "Tooth-mesh frequency [Hz]."
        return abs(n1) / 60 * self.pinion.z

    def pitch_line_velocity(self, n1: float) -> float:
        "Pitch line velocity [m/s] at the operating pitch circle."
        return math.pi * self.d_w1 * abs(n1) / 60_000

    def forces(self, T1: float) -> dict[str, float]:
        "Tooth forces [N] for pinion torque T1 [N·m] (see above), and the pitch radii [mm] they act at."
        Ft = 2000 * T1 / self.d_w1
        return {"F_t": Ft, "F_r": Ft * math.tan(self.mesh.alpha_wt), "F_a": Ft * math.tan(self.mesh.beta_w),
                "r_w1": self.d_w1 / 2, "r_w2": self.d_w2 / 2}

    # --- checks -------------------------------------------------------------------
    def problems(self) -> list[str]:
        "Readable list of the geometry checks that fail."
        p = []
        mc = self.mesh
        for g in (self.pinion, self.gear):
            if g.undercut: p.append(f"{g.name} is undercut (x = {g.x:+.3f})")
            if g.san <= 0.25 * g.mn: p.append(f"{g.name} tip is too thin (s_an = {g.san:.3f} mm < 0.25 m_n)")
        if mc.eps_alpha < mc.eps_alpha_min:
            p.append(f"transverse contact ratio {mc.eps_alpha:.2f} < {mc.eps_alpha_min}")
        if mc.interference: p.append("tip/root interference")
        if min(mc.clearance12, mc.clearance21) <= 0: p.append("no tip clearance")
        return p

    @property
    def ok(self) -> bool:
        return not self.problems()

    def warnings(self) -> list[str]:
        "Advisory notes: legal but worth a second look."
        w = []
        mc = self.mesh
        if 0 < self.pinion.beta and mc.eps_beta < 1:
            w.append(f"overlap ratio ε_β = {mc.eps_beta:.2f} < 1: part of the helical smoothness is lost")
        if self.pinion.beta > math.radians(30):
            w.append("helix angle above 30°: high axial thrust, consider a double helical gear")
        if self.pinion.b / self.pinion.d > 1.2:
            w.append(f"face width / pinion diameter = {self.pinion.b / self.pinion.d:.2f} > 1.2: load spreads poorly")
        if self.pinion.san < 0.4 * self.pinion.mn:
            w.append(f"pinion tip s_an = {self.pinion.san / self.pinion.mn:.2f} m_n < 0.4 m_n: brittle when case hardened")
        return w

    def summary(self) -> str:
        "Short text summary of the pair."
        p, g, mc = self.pinion, self.gear, self.mesh
        lines = [f"u = {self.ratio:.4f} ({p.z}/{g.z}), m_n = {p.mn:g} mm, β = {math.degrees(p.beta):.3f}° "
                 f"({p.hand}/{g.hand}), α_n = {math.degrees(p.alpha_n):g}°, a_w = {self.a_w:.4f} mm",
                 f"  {'gear':7s} {'z':>4s} {'x':>7s} {'k*':>6s} {'d':>9s} {'da':>9s} {'df':>9s} {'b':>6s}"]
        for x in (p, g):
            lines.append(f"  {x.name:7s} {x.z:4d} {x.x:+7.3f} {x.k_tip:6.3f} {x.d:9.3f} {x.da:9.3f} {x.df:9.3f} {x.b:6.1f}")
        lines.append(f"  ε_α {mc.eps_alpha:.3f}, ε_β {mc.eps_beta:.3f}, ε_γ {mc.eps_gamma:.3f}, "
                     f"α_wt {math.degrees(mc.alpha_wt):.3f}°, β_w {math.degrees(mc.beta_w):.3f}°")
        lines.append("  checks: " + ("all pass" if self.ok else "; ".join(self.problems())))
        if self.warnings():
            lines.append("  notes: " + "; ".join(self.warnings()))
        return "\n".join(lines)

In [ ]:
pair = HelicalPair.build(17, 52, mn=1.5, beta=math.radians(15), b=20)
print(pair.summary())
f = pair.forces(40.0)
print({k: round(v, 1) for k, v in f.items()}, f"mesh {pair.mesh_frequency(3000):.0f} Hz")

In [ ]:
#| hide
assert pair.ok and pair.gear.hand == "left"
assert math.isclose(pair.a_w, tidy_centre_distance((17 + 52) * 1.5 / math.cos(math.radians(15)) / 2), abs_tol=1e-6)
assert abs(pair.a_w * 2 - round(pair.a_w * 2)) < 1e-6                       # tidy 0.5 mm step
# fixed centre distance is reached
_p = HelicalPair.build(17, 52, mn=1.5, beta=math.radians(15), b=20, a_w=54.0)
assert abs(_p.a_w - 54.0) < 1e-6
# torque balance: F_t r_w1 = T1, and gear torque = T1 u
_f = _p.forces(10.0)
assert math.isclose(_f["F_t"] * _f["r_w1"] / 1000, 10.0)
assert math.isclose(_f["F_a"], _f["F_t"] * math.tan(_p.mesh.beta_w))
assert _p.speeds(3000)["gear"] < 0
# tip shortening keeps the clearance at c* = 0.25
assert abs(_p.mesh.clearance12 - 0.25 * 1.5) < 1e-6 or _p.pinion.k_tip == 0
# a narrow face with a small helix flags the overlap
assert any("ε_β" in w for w in HelicalPair.build(17, 52, 1.5, math.radians(8), b=6).warnings())

## References

1. ISO 21771:2007, *Gears — Cylindrical involute gears and gear pairs — Concepts and geometry*.
2. ISO/TR 4467:1982, *Addendum modification of the teeth of cylindrical gears for speed-reducing and speed-increasing gear pairs*.
3. DIN 3992:1964, *Profilverschiebung bei Stirnrädern mit Außenverzahnung*.
4. G. Niemann, H. Winter, *Maschinenelemente*, Band II, Springer, 2nd ed., 1989 (shift split, helix angle choice).
5. J. R. Davis (ed.), *Gear Materials, Properties, and Manufacture*, ASM International, 2005 (helix angle and noise).